In [1]:
# =====================================================================
# CÉLULA 1 — Importações
# =====================================================================
import os
import re
import sys
import unicodedata
from datetime import date, datetime, time, timedelta
from pathlib import Path

import pandas as pd

ENCODINGS_TENTATIVA = ("utf-8-sig", "utf-8", "latin-1", "cp1252")
SEPARADORES_TENTATIVA = (";", ",", "\t", "|")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

In [2]:
# =====================================================================
# CÉLULA 2 — Caminhos dinâmicos
# =====================================================================
def get_base_dir() -> Path:
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent

    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        return Path(vsc).resolve().parent

    arquivo = globals().get("__file__")
    if arquivo:
        return Path(arquivo).resolve().parent

    return Path.cwd()


BASE_DIR = get_base_dir()


def resolver_pasta_arquivos(nome: str = "arquivos") -> Path:
    """1. AFD_PASTA  2. <base>/arquivos  3. <_MEIPASS>/arquivos  4. cria"""
    env = os.getenv("AFD_PASTA")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]
    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_dir():
            return candidato

    destino = BASE_DIR / nome
    destino.mkdir(parents=True, exist_ok=True)
    return destino


PASTA_ARQUIVOS = resolver_pasta_arquivos()

print("BASE_DIR       :", BASE_DIR)
print("PASTA_ARQUIVOS :", PASTA_ARQUIVOS)

BASE_DIR       : \\AGROSRV\Automacoes\Marcacao_Pontos
PASTA_ARQUIVOS : \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos


In [ ]:
# =====================================================================
# CÉLULA 3 — Arquivos CSV encontrados + leitura do relatório RHiD
# =====================================================================
EXTENSOES_CSV = {".csv", ".txt"}


def parece_relatorio_rhid(caminho: Path) -> bool:
    """Confere se a 1ª linha tem cara de cabeçalho do relatório do RHiD."""
    for enc in ENCODINGS_TENTATIVA:
        try:
            with caminho.open("r", encoding=enc, errors="strict") as f:
                cabecalho = f.readline().lower()
            break
        except UnicodeDecodeError:
            continue
    else:
        return False

    return ("cpf" in cabecalho or "funcionario" in cabecalho) and "dia" in cabecalho


def listar_relatorios(pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or PASTA_ARQUIVOS)
    if not pasta.is_dir():
        return []
    return [
        item for item in sorted(pasta.iterdir())
        if item.is_file()
        and item.suffix.lower() in EXTENSOES_CSV
        and parece_relatorio_rhid(item)
    ]


def detectar_encoding(caminho: Path) -> str:
    """Testa os encodings na ordem e devolve o primeiro que decodifica."""
    for enc in ENCODINGS_TENTATIVA:
        try:
            with caminho.open("r", encoding=enc, errors="strict") as f:
                f.read(4096)
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"


def detectar_separador(caminho: Path, encoding: str) -> str:
    """Conta ocorrências na 1ª linha e escolhe o separador mais provável."""
    with caminho.open("r", encoding=encoding, errors="replace") as f:
        cabecalho = f.readline()

    contagem = {sep: cabecalho.count(sep) for sep in SEPARADORES_TENTATIVA}
    melhor = max(contagem, key=contagem.get)
    return melhor if contagem[melhor] > 0 else ";"


def _parse_datas(serie: pd.Series) -> pd.Series:
    """Converte datas tentando os formatos mais comuns do RHiD."""
    formatos = ("%d/%m/%Y", "%d/%m/%y", "%d-%m-%Y", "%Y-%m-%d", "%d.%m.%Y")

    for fmt in formatos:
        convertido = pd.to_datetime(serie, format=fmt, errors="coerce")
        if convertido.notna().any():
            return convertido

    return pd.to_datetime(serie, dayfirst=True, errors="coerce")


def ler_relatorio(caminho: Path | str) -> pd.DataFrame:
    """
    Lê o CSV exportado do RHiD.

    - Detecta encoding e separador automaticamente
    - Normaliza nomes de colunas (NFKD + snake_case)
    - Converte a coluna de data com formatos explícitos
    - Mantém tudo como texto para não perder zeros à esquerda (CPF/PIS)
    """
    caminho = Path(caminho)
    encoding = detectar_encoding(caminho)
    separador = detectar_separador(caminho, encoding)

    df = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        dtype=str,
        keep_default_na=False,
        skipinitialspace=True,
    )

    df.columns = [normalizar_nome_coluna(c) for c in df.columns]

    # Remove colunas totalmente vazias (comuns em exports do RHiD)
    df = df.loc[:, (df != "").any(axis=0)]

    # Converte a coluna de data (qualquer sinônimo de 'dia')
    colunas_data = ("dia", "data", "data_dia")
    col_data = next((c for c in colunas_data if c in df.columns), None)
    if col_data:
        df[col_data] = _parse_datas(df[col_data])

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()

    return df.reset_index(drop=True)


ARQUIVOS = listar_relatorios()
print(f"{len(ARQUIVOS)} arquivo(s) em {PASTA_ARQUIVOS}")
for arq in ARQUIVOS:
    print(f"  - {arq.name}  ({arq.stat().st_size / 1024:.1f} KB)")

1 arquivo(s) em \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
  - 2026 - Outubro.CSV  (58.4 KB)


In [ ]:
# =====================================================================
# CÉLULA 4 — Leiautes, colunas e parâmetros legais
# =====================================================================
CAMPOS_FIXOS = {
    "nome": ("nome_do_funcionario", "nome_funcionario", "funcionario", "nome", "colaborador"),
    "cpf": ("cpf_do_funcionario", "cpf_funcionario", "cpf", "cpf_colaborador"),
    "dia": ("dia", "data", "data_dia"),
    "previsto": ("previsto", "horario_previsto", "jornada_prevista", "escala", "horario_contratual"),
    "total_normais": ("total_normais", "normais", "total_normal"),
    "total_noturno": ("total_noturno", "noturno"),
    "dia_falta": ("dia_falta", "falta_dia"),
    "falta_e_atraso": ("falta_e_atraso", "faltas_e_atrasos", "falta_atraso"),
    "abono": ("abono", "abonos"),
    "extra_0_d": ("extra_0_d", "extra_0d"),
    "extra_60_d": ("extra_60_d", "extra_60d"),
    "extra_diurna": ("extra_diurna", "extras_diurna"),
    "extra_noturna": ("extra_noturna", "extras_noturna"),
    "banco_total": ("banco_total", "total_banco", "banco_dia"),
    "banco_saldo": ("banco_saldo", "saldo_banco", "saldo"),
    "justificativas": ("justificativas", "justificativa", "observacoes", "obs"),
}

CAMPOS_DURACAO = (
    "total_normais", "total_noturno", "dia_falta", "falta_e_atraso", "abono",
    "extra_0_d", "extra_60_d", "extra_diurna", "extra_noturna",
    "banco_total", "banco_saldo",
)

DIAS_ABREV = ("seg", "ter", "qua", "qui", "sex", "sáb", "dom")

# Intervalo NÃO remunerado após cada par de marcação:
#   após o par 1 (Saída 1 -> Entrada 2) = Café
#   após o par 2 (Saída 2 -> Entrada 3) = Almoço
ROTULO_INTERVALO = {1: "Café", 2: "Almoço", 3: "Intervalo 3", 4: "Intervalo 4"}

# ---------------------------------------------------------------------
# PARÂMETROS LEGAIS — CLT + CCT Bragança Paulista/Joanópolis (SP011470/2025)
# ---------------------------------------------------------------------
PARAMETROS_LEGAIS = {
    # CLT art. 58, §1º
    "tolerancia_min_por_marcacao": 5,
    "tolerancia_min_por_dia": 10,

    # CLT art. 71, caput
    "jornada_longa": timedelta(hours=6),
    "jornada_media": timedelta(hours=4),
    "intervalo_legal_longo": timedelta(hours=1),
    "intervalo_legal_medio": timedelta(minutes=15),

    # CLT art. 611-A, III + Tema 1046 STF (CCT pode reduzir até 30 min)
    "usar_minimo_cct": False,
    "intervalo_minimo_cct": timedelta(minutes=30),

    # CLT art. 73, §§1º e 2º + Súmula 60 TST
    "noturno_inicio": time(22, 0),
    "noturno_fim": time(5, 0),
    "fator_hora_noturna": 52.5,
    "adicional_noturno": 0.20,
    "aplicar_sumula_60": True,

    # CLT art. 59, caput
    "limite_extra_dia": timedelta(hours=2),

    # CLT art. 71, §4º
    "percentual_intervalo_suprimido": 0.50,

    # CCT cláusula 20 — adicional de hora extra majorado
    "adicional_hora_extra": 0.60,   # 60% (CLT seria 50%)

    # CCT cláusula 32 — banco de horas
    "banco_limite_mensal": timedelta(hours=40),
    "banco_prazo_compensacao_dias": 90,
    "banco_vedado_domingo_feriado": True,

    # CCT cláusulas 47/48 — domingos e feriados
    "jornada_max_domingo_feriado": timedelta(hours=8),
    "adicional_feriado": 1.00,
    "indenizacao_feriado_fracao": 1 / 30,

    # CCT cláusula 17 — caixa
    "quebra_caixa_mensal": 108.44,

    # EC 141/2026 — transição da jornada semanal
    "jornada_semanal_por_data": [
        (date(2026, 12, 15), timedelta(hours=42)),
        (date(2028, 2, 15), timedelta(hours=40)),
    ],
    "jornada_semanal_padrao": timedelta(hours=44),

    # CLT art. 66 — interjornada
    "interjornada_minima": timedelta(hours=11),

    # Alerta de marcação muito distante do previsto
    "alerta_desvio_min": 60,
}


def normalizar_nome_coluna(nome) -> str:
    texto_ = str(nome or "").strip().lower()
    texto_ = unicodedata.normalize("NFKD", texto_)
    texto_ = "".join(c for c in texto_ if not unicodedata.combining(c))
    texto_ = re.sub(r"[^a-z0-9]+", "_", texto_)
    return texto_.strip("_")


def resolver_colunas(colunas) -> dict[str, str]:
    mapa_normalizado = {normalizar_nome_coluna(c): c for c in colunas}
    resolvido = {}

    for campo, candidatos in CAMPOS_FIXOS.items():
        for candidato in candidatos:
            if candidato in mapa_normalizado:
                resolvido[campo] = mapa_normalizado[candidato]
                break
        else:
            for candidato in candidatos:
                achado = next(
                    (real for norm, real in mapa_normalizado.items()
                     if norm.startswith(candidato)), None)
                if achado:
                    resolvido[campo] = achado
                    break

    return resolvido


def mapear_pares(colunas) -> dict[int, dict[str, str]]:
    pares: dict[int, dict[str, str]] = {}

    for coluna in colunas:
        norm = normalizar_nome_coluna(coluna)

        achado = re.match(r"^(entrada|ent)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["entrada"] = coluna
            continue

        achado = re.match(r"^(saida|sai)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["saida"] = coluna

    return dict(sorted(pares.items()))


def conferir_leiaute(df: pd.DataFrame) -> tuple[dict, dict, list[str]]:
    campos = resolver_colunas(df.columns)
    pares = mapear_pares(df.columns)

    problemas = []
    for obrigatorio in ("nome", "cpf", "dia"):
        if obrigatorio not in campos:
            problemas.append(
                f"coluna obrigatória não encontrada: {obrigatorio}")

    if not pares:
        problemas.append("nenhum par entrada/saída encontrado")

    return campos, pares, problemas


def jornada_semanal_vigente(data_ref: date) -> timedelta:
    """Retorna a jornada semanal máxima vigente na data (EC 141/2026)."""
    for marco, carga in PARAMETROS_LEGAIS["jornada_semanal_por_data"]:
        if data_ref >= marco:
            return carga
    return PARAMETROS_LEGAIS["jornada_semanal_padrao"]


# Cargos do Funcionarios.csv e suas particularidades
CARGOS = {
    "motorista": {
        "intervalo_minimo": timedelta(hours=1),
        "interjornada": timedelta(hours=11),
        "cct_motoristas": "MR064475/2026",
        "permite_hora_extra": True,
    },
    "ajudante de caminhao": {
        "intervalo_minimo": timedelta(hours=1),
        "interjornada": timedelta(hours=11),
        "cct_motoristas": "MR064475/2026",
        "permite_hora_extra": True,
    },
    "caixa": {
        "quebra_caixa": 108.44,
        "permite_hora_extra": True,
    },
    "auxiliar do comercio": {"permite_hora_extra": True},
    "chefe de setor": {"permite_hora_extra": True},
}


def normalizar_cargo(cargo: str) -> str:
    return normalizar_nome_coluna(cargo).replace("_", " ")


def regras_do_cargo(cargo: str) -> dict:
    base = {
        "intervalo_minimo": PARAMETROS_LEGAIS["intervalo_legal_longo"],
        "interjornada": PARAMETROS_LEGAIS["interjornada_minima"],
        "permite_hora_extra": True,
    }
    base.update(CARGOS.get(normalizar_cargo(cargo), {}))
    return base


def carregar_funcionarios(caminho: Path | None = None) -> pd.DataFrame:
    """Lê o Funcionarios.csv e devolve cpf (só dígitos) + cargo + departamento."""
    caminho = Path(caminho or (PASTA_ARQUIVOS / "Funcionarios.csv"))
    if not caminho.is_file():
        return pd.DataFrame(columns=["cpf", "cargo", "departamento"])

    df = ler_relatorio(caminho)

    # Resolve o nome real da coluna de CPF entre os sinônimos
    campos = resolver_colunas(df.columns)
    col_cpf = campos.get("cpf")
    if col_cpf:
        df = df.rename(columns={col_cpf: "cpf"})
        df["cpf"] = df["cpf"].astype(str).str.replace(r"\D", "", regex=True)
    else:
        df["cpf"] = ""

    return df


def enriquecer_com_cargo(df: pd.DataFrame,
                         funcionarios: pd.DataFrame,
                         campos: dict | None = None) -> pd.DataFrame:
    """Adiciona cargo e departamento ao relatório de ponto, casando por CPF."""
    df = df.copy()

    # Resolve o nome real da coluna de CPF no relatório de ponto
    campos = campos or resolver_colunas(df.columns)
    col_cpf = campos.get("cpf")

    if not col_cpf:
        df["cargo"] = ""
        df["departamento"] = ""
        return df

    df = df.rename(columns={col_cpf: "cpf"})
    df["cpf"] = df["cpf"].astype(str).str.replace(r"\D", "", regex=True)

    if funcionarios.empty or "cpf" not in funcionarios.columns:
        df["cargo"] = ""
        df["departamento"] = ""
        return df

    mapa = funcionarios.drop_duplicates("cpf").set_index("cpf")
    df["cargo"] = df["cpf"].map(mapa["cargo"] if "cargo" in mapa else {})
    df["departamento"] = (
        df["cpf"].map(mapa["departamento"] if "departamento" in mapa else {})
    )
    df["cargo"] = df["cargo"].fillna("").astype(str)
    df["departamento"] = df["departamento"].fillna("").astype(str)
    return df

In [5]:
# =====================================================================
# CÉLULA 5 — Detecção de intervalo, jornada, noturno e Súmula 60
# =====================================================================
def _combinar(dia: date, hora: time) -> datetime:
    return datetime.combine(dia, hora)


def _ler_hora(valor) -> time | None:
    if valor is None or pd.isna(valor):
        return None
    if isinstance(valor, time):
        return valor
    if isinstance(valor, datetime):
        return valor.time()
    texto = str(valor).strip()
    if not texto:
        return None
    for fmt in ("%H:%M", "%H:%M:%S", "%H%M"):
        try:
            return datetime.strptime(texto, fmt).time()
        except ValueError:
            continue
    return None


def detectar_intervalo_almoco(pares: dict[int, dict[str, str]],
                              linha: pd.Series) -> int | None:
    """
    Identifica qual par entrada/saída corresponde ao intervalo intrajornada.

    1. Maior duração entre a saída de um par e a entrada do próximo
    2. Desempate: o intervalo que cruza o meio-dia (11h-15h)
    3. Fallback: None (sem intervalo detectado)

    Retorna o índice do par cuja SAÍDA inicia o intervalo.
    """
    candidatos = []
    indices = sorted(pares)

    for pos, idx in enumerate(indices[:-1]):
        saida = _ler_hora(linha.get(pares[idx].get("saida")))
        proximo = indices[pos + 1]
        entrada = _ler_hora(linha.get(pares[proximo].get("entrada")))
        if saida is None or entrada is None:
            continue

        dt_saida = datetime.combine(date.today(), saida)
        dt_entrada = datetime.combine(date.today(), entrada)
        if dt_entrada < dt_saida:
            dt_entrada += timedelta(days=1)
        duracao = dt_entrada - dt_saida

        cruza_meio_dia = saida <= time(15, 0) and entrada >= time(11, 0)

        candidatos.append({
            "indice": idx,
            "duracao": duracao,
            "cruza_meio_dia": cruza_meio_dia,
        })

    if not candidatos:
        return None

    candidatos.sort(
        key=lambda c: (c["duracao"], c["cruza_meio_dia"]), reverse=True
    )
    melhor = candidatos[0]

    if melhor["duracao"] < timedelta(minutes=15):
        return None

    return melhor["indice"]


def validar_intervalo(indice: int | None, duracao: timedelta | None,
                      regra_cargo: dict) -> list[str]:
    """Aplica CLT art. 71 sobre o intervalo detectado."""
    avisos = []
    minimo = regra_cargo.get(
        "intervalo_minimo", PARAMETROS["intervalo_minimo"])

    if indice is None or duracao is None:
        avisos.append("intervalo intrajornada não identificado")
        return avisos

    if duracao < minimo:
        suprimido = minimo - duracao
        avisos.append(
            f"intervalo suprimido: {duracao} concedido, mínimo {minimo} — "
            f"suprimir {suprimido} "
            f"(+{PARAMETROS['percentual_intervalo_suprimido']:.0%} indenizatório)"
        )
    elif duracao > PARAMETROS["intervalo_maximo"]:
        avisos.append(
            f"intervalo de {duracao} excede o máximo de "
            f"{PARAMETROS['intervalo_maximo']} (CCT cláusula 47)"
        )

    return avisos


def _minutos_no_periodo_noturno(inicio: datetime, fim: datetime) -> int:
    """Conta minutos de relógio dentro de 22h-5h entre dois instantes."""
    total = 0
    cursor = inicio
    while cursor < fim:
        hora = cursor.time()
        noturno = hora >= PARAMETROS["noturno_inicio"] or hora < PARAMETROS["noturno_fim"]
        proximo = min(cursor + timedelta(minutes=1), fim)
        if noturno:
            total += int((proximo - cursor).total_seconds() // 60)
        cursor = proximo
    return total


def calcular_jornada_dia(linha: pd.Series, pares: dict[int, dict[str, str]],
                         indice_intervalo: int | None,
                         regra_cargo: dict) -> dict:
    """
    Calcula as horas de um dia.

    Retorna duração bruta, intervalo, horas noturnas (fator 52min30s)
    e horas prorrogadas pós-5h (Súmula 60).
    """
    dia = pd.to_datetime(linha.get("dia"), dayfirst=True, errors="coerce")
    if pd.isna(dia):
        return {"erro": "data inválida"}
    dia = dia.date()

    marcas: list[datetime] = []
    for idx in sorted(pares):
        for tipo in ("entrada", "saida"):
            coluna = pares[idx].get(tipo)
            if not coluna:
                continue
            hora = _ler_hora(linha.get(coluna))
            if hora is None:
                continue
            marcas.append(_combinar(dia, hora))

    if len(marcas) < 2:
        return {"erro": "marcações insuficientes"}

    marcas.sort()
    for i in range(1, len(marcas)):
        if marcas[i] < marcas[i - 1]:
            marcas[i] += timedelta(days=1)

    # Intervalo intrajornada
    intervalo = timedelta(0)
    if indice_intervalo is not None:
        saida = _ler_hora(
            linha.get(pares.get(indice_intervalo, {}).get("saida")))
        entrada = _ler_hora(
            linha.get(pares.get(indice_intervalo + 1, {}).get("entrada")))
        if saida and entrada:
            dt_s = _combinar(dia, saida)
            dt_e = _combinar(dia, entrada)
            if dt_e < dt_s:
                dt_e += timedelta(days=1)
            intervalo = dt_e - dt_s

    bruto = marcas[-1] - marcas[0]
    liquido = bruto - intervalo

    # Horas noturnas (22h-5h) com fator de redução
    minutos_noturnos = 0
    for i in range(len(marcas) - 1):
        minutos_noturnos += _minutos_no_periodo_noturno(
            marcas[i], marcas[i + 1])

    horas_noturnas = minutos_noturnos / PARAMETROS["fator_hora_noturna"]

    # Súmula 60: prorrogação após as 5h
    prorrogacao_noturna = timedelta(0)
    if PARAMETROS["aplicar_sumula_60"]:
        inicio_noturno = (
            marcas[0].time() >= PARAMETROS["noturno_inicio"]
            or marcas[0].time() < PARAMETROS["noturno_fim"]
        )
        if inicio_noturno:
            for i in range(len(marcas) - 1):
                ini, fim = marcas[i], marcas[i + 1]
                limite = _combinar(fim.date(), PARAMETROS["noturno_fim"])
                if ini < limite < fim:
                    prorrogacao_noturna += fim - limite

    return {
        "dia": dia,
        "primeira_marcacao": marcas[0].time(),
        "ultima_marcacao": marcas[-1].time(),
        "duracao_bruta": bruto,
        "intervalo": intervalo,
        "duracao_liquida": liquido,
        "minutos_noturnos_relogio": minutos_noturnos,
        "horas_noturnas": round(horas_noturnas, 4),
        "prorrogacao_noturna": prorrogacao_noturna,
        "marcacoes": len(marcas),
    }

In [6]:
# =====================================================================
# CÉLULA 6 — Banco de horas (CCT cláusula 32) e consolidação
# =====================================================================
def parse_previsto(valor) -> timedelta | None:
    """Aceita '08:00', '8h', '8h20', '08:00-17:00' (usa o 1º horário)."""
    if valor is None:
        return None
    texto = str(valor).strip()
    if not texto:
        return None

    if "-" in texto:
        texto = texto.split("-")[0].strip()

    m = re.match(r"^(\d{1,2}):(\d{2})$", texto)
    if m:
        return timedelta(hours=int(m.group(1)), minutes=int(m.group(2)))

    m = re.match(r"^(\d{1,2})\s*h\s*(\d{2})?$", texto, re.IGNORECASE)
    if m:
        return timedelta(hours=int(m.group(1)), minutes=int(m.group(2) or 0))

    m = re.match(r"^(\d{1,2})(?:[.,](\d{1,2}))?$", texto)
    if m:
        horas = int(m.group(1))
        minutos = int((m.group(2) or "0").ljust(2, "0"))
        return timedelta(hours=horas, minutes=minutos)

    return None


def calcular_banco_dia(jornada: dict, previsto: timedelta | None,
                       regra_cargo: dict, dia_semana: int) -> dict:
    """
    Compara o trabalhado com o previsto e classifica o saldo.

    CCT cláusula 32:
      - limite de 40h compensatórias por mês
      - compensação em até 90 dias
      - vedada compensação em domingos e feriados
    """
    if "erro" in jornada:
        return {"erro": jornada["erro"]}

    trabalhado = jornada["duracao_liquida"]
    previsto = previsto or timedelta(0)
    saldo = trabalhado - previsto

    eh_domingo = dia_semana == 6
    vedado = eh_domingo or jornada.get("feriado", False)

    if vedado and saldo > timedelta(0):
        return {
            "saldo": saldo,
            "tipo": "feriado_100",
            "compensavel": False,
            "observacao": "trabalho em domingo/feriado — pagamento 100%, "
                          "vedada compensação (CCT cl. 32 'e')",
        }

    if saldo > timedelta(0):
        excedente = max(timedelta(0), saldo - PARAMETROS["limite_extra_dia"])
        return {
            "saldo": saldo,
            "tipo": "credito",
            "compensavel": True,
            "excedente_limite": excedente,
            "observacao": (
                f"excede limite de 2h/dia em {excedente} (CLT art. 59)"
                if excedente > timedelta(0) else ""
            ),
        }

    if saldo < timedelta(0):
        return {
            "saldo": saldo,
            "tipo": "debito",
            "compensavel": True,
            "observacao": "débito — verificar justificativa/abono",
        }

    return {"saldo": timedelta(0), "tipo": "neutro", "compensavel": True}


def consolidar_banco_mensal(registros: list[dict]) -> dict:
    """Aplica o teto de 40h/mês da CCT cláusula 32."""
    credito = sum(
        (r["saldo"] for r in registros
         if r.get("tipo_saldo") == "credito" and r.get("saldo")),
        timedelta(0),
    )
    debito = sum(
        (r["saldo"] for r in registros
         if r.get("tipo_saldo") == "debito" and r.get("saldo")),
        timedelta(0),
    )
    saldo = credito + debito
    teto = PARAMETROS["banco_limite_mensal"]

    return {
        "credito_mes": credito,
        "debito_mes": debito,
        "saldo_mes": saldo,
        "teto_cct": teto,
        "excede_teto": credito > teto,
        "credito_compensavel": min(credito, teto),
        "credito_excedente": max(timedelta(0), credito - teto),
        "prazo_compensacao_dias": PARAMETROS["banco_prazo_compensacao_dias"],
    }

In [7]:
# =====================================================================
# CÉLULA 7 — Pipeline: processa o relatório e exporta
# =====================================================================
def processar_relatorio(caminho: Path,
                        funcionarios: pd.DataFrame | None = None) -> dict:
    df = ler_relatorio(caminho)

    if funcionarios is not None:
        df = enriquecer_com_cargo(df, funcionarios)

    campos, pares, problemas = conferir_leiaute(df)
    if problemas:
        return {"arquivo": caminho.name, "erros": problemas}

    registros = []
    for _, linha in df.iterrows():
        cargo = linha.get("cargo", "")
        regra = regras_do_cargo(str(cargo))

        indice_intervalo = detectar_intervalo_almoco(pares, linha)
        jornada = calcular_jornada_dia(linha, pares, indice_intervalo, regra)

        if "erro" in jornada:
            registros.append(
                {"dia": linha.get("dia"), "erro": jornada["erro"]})
            continue

        avisos = validar_intervalo(
            indice_intervalo, jornada["intervalo"], regra)

        col_previsto = campos.get("previsto")
        previsto = parse_previsto(
            linha.get(col_previsto)) if col_previsto else None

        dia_semana = jornada["dia"].weekday()
        banco = calcular_banco_dia(jornada, previsto, regra, dia_semana)

        registros.append({
            "nome": linha.get(campos.get("nome", ""), ""),
            "cpf": linha.get(campos.get("cpf", ""), ""),
            "cargo": cargo,
            "dia": jornada["dia"],
            "dia_semana": DIAS_ABREV[dia_semana],
            "entrada": jornada["primeira_marcacao"],
            "saida": jornada["ultima_marcacao"],
            "bruto": jornada["duracao_bruta"],
            "intervalo": jornada["intervalo"],
            "liquido": jornada["duracao_liquida"],
            "previsto": previsto,
            "horas_noturnas": jornada["horas_noturnas"],
            "prorrogacao_noturna": jornada["prorrogacao_noturna"],
            "saldo": banco.get("saldo"),
            "tipo_saldo": banco.get("tipo"),
            "avisos": "; ".join(
                [a for a in avisos + [banco.get("observacao", "")] if a]
            ),
        })

    return {
        "arquivo": caminho.name,
        "registros": registros,
        "banco": consolidar_banco_mensal(registros),
    }


def exportar_excel(resultado: dict, destino: Path | None = None) -> Path:
    destino = destino or (
        PASTA_ARQUIVOS / f"{Path(resultado['arquivo']).stem}_analisado.xlsx")

    df = pd.DataFrame(resultado["registros"])
    banco = pd.DataFrame([resultado["banco"]]).T.reset_index()
    banco.columns = ["campo", "valor"]

    with pd.ExcelWriter(destino, engine="openpyxl",
                        datetime_format="dd/mm/yyyy hh:mm:ss") as writer:
        banco.to_excel(writer, sheet_name="resumo", index=False)
        if not df.empty:
            df.to_excel(writer, sheet_name="marcacoes", index=False)

    return destino


FUNCIONARIOS = carregar_funcionarios()
RESULTADOS = []

for arquivo in ARQUIVOS:
    print()
    resultado = processar_relatorio(arquivo, FUNCIONARIOS)
    RESULTADOS.append(resultado)

    if "erros" in resultado:
        print(f"{arquivo.name}: {resultado['erros']}")
        continue

    print(f"{arquivo.name}: {len(resultado['registros'])} registro(s)")
    print(f"  banco: {resultado['banco']['saldo_mes']} "
          f"(teto {resultado['banco']['teto_cct']})")
    print("  planilha:", exportar_excel(resultado).name)

C:\Users\Home\AppData\Local\Temp\ipykernel_19592\3282542025.py:84: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["dia"] = pd.to_datetime(df["dia"], dayfirst=True, errors="coerce")


KeyError: 'cpf'

In [ ]:
# =====================================================================
# CÉLULA 8 — Banco de horas (CCT cláusula 32) e consolidação
# =====================================================================
def calcular_banco_dia(jornada: dict, previsto: timedelta | None,
                       regra_cargo: dict, dia_semana: int) -> dict:
    """
    Compara o trabalhado com o previsto e classifica o saldo.

    CCT cláusula 32:
      - limite de 40h compensatórias por mês
      - compensação em até 90 dias
      - vedada compensação em domingos e feriados
    """
    if "erro" in jornada:
        return {"erro": jornada["erro"]}

    trabalhado = jornada["duracao_liquida"]
    previsto = previsto or timedelta(0)
    saldo = trabalhado - previsto

    # Domingo (6) ou feriado: não entra no banco
    eh_domingo = dia_semana == 6
    vedado = eh_domingo or jornada.get("feriado", False)

    if vedado and saldo > timedelta(0):
        return {
            "saldo": saldo,
            "tipo": "feriado_100",
            "compensavel": False,
            "observacao": "trabalho em domingo/feriado — pagamento 100%, "
                          "vedada compensação (CCT cl. 32 'e')",
        }

    if saldo > timedelta(0):
        # Excedente além do limite diário de 2h
        excedente = max(timedelta(0), saldo - PARAMETROS["limite_extra_dia"])
        return {
            "saldo": saldo,
            "tipo": "credito",
            "compensavel": True,
            "excedente_limite": excedente,
            "observacao": (
                f"excede limite de 2h/dia em {excedente} (CLT art. 59)"
                if excedente > timedelta(0) else ""
            ),
        }

    if saldo < timedelta(0):
        return {
            "saldo": saldo,
            "tipo": "debito",
            "compensavel": True,
            "observacao": "débito — verificar justificativa/abono",
        }

    return {"saldo": timedelta(0), "tipo": "neutro", "compensavel": True}


def consolidar_banco_mensal(registros: list[dict]) -> dict:
    """Aplica o teto de 40h/mês da CCT cláusula 32."""
    credito = sum(
        (r["banco"]["saldo"] for r in registros
         if r.get("banco", {}).get("tipo") == "credito"),
        timedelta(0),
    )
    debito = sum(
        (r["banco"]["saldo"] for r in registros
         if r.get("banco", {}).get("tipo") == "debito"),
        timedelta(0),
    )
    saldo = credito + debito
    teto = PARAMETROS["banco_limite_mensal"]

    return {
        "credito_mes": credito,
        "debito_mes": debito,
        "saldo_mes": saldo,
        "teto_cct": teto,
        "excede_teto": credito > teto,
        "credito_compensavel": min(credito, teto),
        "credito_excedente": max(timedelta(0), credito - teto),
        "prazo_compensacao_dias": PARAMETROS["banco_prazo_compensacao_dias"],
    }

In [ ]:
# =====================================================================
# CÉLULA 9 — Pipeline: processa o relatório e exporta
# =====================================================================
def processar_relatorio(caminho: Path) -> dict:
    df = ler_relatorio(caminho)          # sua função de leitura
    campos, pares, problemas = conferir_leiaute(df)

    if problemas:
        return {"arquivo": caminho.name, "erros": problemas}

    registros = []
    for _, linha in df.iterrows():
        cargo = linha.get(campos.get("cargo", ""), "")
        regra = regras_do_cargo(str(cargo))

        indice_intervalo = detectar_intervalo_almoco(pares, linha)
        jornada = calcular_jornada_dia(linha, pares, indice_intervalo, regra)

        if "erro" in jornada:
            registros.append({"dia": linha.get("dia"),
                              "erro": jornada["erro"]})
            continue

        avisos = validar_intervalo(
            indice_intervalo, jornada["intervalo"], regra
        )

        previsto = pd.to_timedelta(
            str(linha.get(campos.get("previsto", ""), "")), errors="coerce"
        )
        dia_semana = jornada["dia"].weekday()
        banco = calcular_banco_dia(jornada, previsto, regra, dia_semana)

        registros.append({
            "nome": linha.get(campos.get("nome", "")),
            "cpf": linha.get(campos.get("cpf", "")),
            "cargo": cargo,
            "dia": jornada["dia"],
            "dia_semana": DIAS_ABREV[dia_semana],
            "entrada": jornada["primeira_marcacao"],
            "saida": jornada["ultima_marcacao"],
            "bruto": jornada["duracao_bruta"],
            "intervalo": jornada["intervalo"],
            "liquido": jornada["duracao_liquida"],
            "previsto": previsto,
            "horas_noturnas": jornada["horas_noturnas"],
            "prorrogacao_noturna": jornada["prorrogacao_noturna"],
            "saldo": banco.get("saldo"),
            "tipo_saldo": banco.get("tipo"),
            "avisos": "; ".join(avisos + [banco.get("observacao", "")]).strip("; "),
        })

    return {
        "arquivo": caminho.name,
        "registros": registros,
        "banco": consolidar_banco_mensal(registros),
    }


RESULTADOS = [processar_relatorio(a) for a in ARQUIVOS]